In [1]:
# %% [markdown]
# # 02_clean.ipynb
# ## 股票和指数数据清洗与合并流程

# %%
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import time

# 设置显示选项
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 创建输出目录
Path("data/clean").mkdir(parents=True, exist_ok=True)

# %% [markdown]
# ## 1. 缺失值检测

# %%
# 读取所有股票CSV文件
stock_files = glob.glob("data/stock/stock_*.csv")
print(f"找到 {len(stock_files)} 个股票文件")

# 存储缺失值统计结果
missing_stats = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file)
    
    # 统计每列缺失值
    missing_count = df.isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    
    missing_stats[code] = pd.DataFrame({
        '列名': missing_count.index,
        '缺失数量': missing_count.values,
        '缺失比例(%)': missing_pct.values
    })

# 合并所有股票的缺失统计
if missing_stats:
    all_missing = pd.concat(missing_stats, keys=missing_stats.keys(), names=['股票代码'])
    print("\n=== 缺失值检测结果 ===")
    print(all_missing[all_missing['缺失数量'] > 0])
    print(f"\n清洗前总缺失值数量: {all_missing['缺失数量'].sum()}")
else:
    print("未找到股票数据文件")

# %% [markdown]
# **清洗说明 - 缺失值检测前后变化：**
# 
# 清洗前：原始CSV文件中可能存在缺失值，影响后续计算和合并操作
# 
# 清洗后：通过统计表格清晰展示了每只股票、每列数据的缺失情况，为后续处理提供依据

# %% [markdown]
# ## 2. 缺失值处理

# %%
# 选择填充策略：使用向前填充(ffill)
# 依据说明：
# 1. 股票日度数据具有时间序列特性，缺失值通常由于停牌或数据中断导致
# 2. 前向填充符合"缺失日的价格延续前一日"的市场逻辑

missing_deleted_records = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file)
    
    original_shape = df.shape
    
    # 记录删除前的缺失情况
    missing_before = df.isnull().sum().sum()
    
    # 使用前向填充处理缺失值
    df = df.ffill()
    
    # 如果第一行仍有缺失，使用后向填充
    df = df.bfill()
    
    # 检查date列是否有NaN
    if df['date'].isnull().any():
        df = df.dropna(subset=['date'])
    
    # 对于价格和成交量列，如果填充后仍有缺失，删除对应行
    price_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    df = df.dropna(subset=price_cols)
    
    # 记录处理后的变化
    missing_after = df.isnull().sum().sum()
    deleted_rows = original_shape[0] - df.shape[0]
    missing_deleted_records[code] = {
        '原始行数': original_shape[0],
        '清洗后行数': df.shape[0],
        '删除行数': deleted_rows,
        '填充缺失值数量': missing_before - missing_after
    }
    
    # 保存回原文件
    df.to_csv(file, index=False)
    print(f"{code}: 删除了 {deleted_rows} 行，填充了 {missing_before - missing_after} 个缺失值")

missing_summary = pd.DataFrame(missing_deleted_records).T
print("\n=== 缺失值处理总结 ===")
print(missing_summary)

# %% [markdown]
# **清洗说明 - 缺失值处理前后变化：**
# 
# 清洗前：存在缺失值的数据行可能无法进行收益率计算和合并操作
# 
# 清洗后：
# - 使用前向填充处理了时间序列中的缺失值
# - 删除了连续缺失严重或关键字段缺失的记录
# - 每只股票的处理情况已记录在上表中

# %% [markdown]
# ## 3. 日期格式统一

# %%
date_conversion_log = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file)
    
    # 记录原始日期格式示例
    original_date_sample = df['date'].iloc[0] if len(df) > 0 else None
    
    # 转换日期格式
    try:
        df['date'] = pd.to_datetime(df['date'])
        date_conversion_log[code] = {
            '原始格式示例': original_date_sample,
            '转换后格式': 'datetime64[ns]',
            '是否成功': True
        }
    except Exception as e:
        date_conversion_log[code] = {
            '原始格式示例': original_date_sample,
            '转换后格式': '转换失败',
            '是否成功': False,
            '错误信息': str(e)
        }
    
    # 设置date为索引
    df.set_index('date', inplace=True)
    
    # 保存回原文件
    df.to_csv(file)
    print(f"{code}: 日期格式已转换为 datetime64，并设为索引")

print("\n=== 日期格式转换日志 ===")
print(pd.DataFrame(date_conversion_log).T)

# %% [markdown]
# **清洗说明 - 日期格式统一前后变化：**
# 
# 清洗前：日期列可能是字符串或object类型，无法进行时间序列操作和索引
# 
# 清洗后：
# - 所有日期列统一为 datetime64 格式，支持时间序列操作
# - 日期已设为DataFrame索引，便于后续对齐和合并

# %% [markdown]
# ## 4. 数据类型检查

# %%
type_check_results = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file, index_col='date', parse_dates=True)
    
    # 需要检查的列
    check_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    type_log = {}
    
    for col in check_cols:
        original_type = df[col].dtype
        type_log[col] = {'原始类型': str(original_type)}
        
        # 检查是否为数值类型
        if not np.issubdtype(df[col].dtype, np.number):
            # 尝试转换为数值类型
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                type_log[col]['转换后类型'] = str(df[col].dtype)
                type_log[col]['转换状态'] = '已转换'
                type_log[col]['转换失败数量'] = df[col].isnull().sum()
            except:
                type_log[col]['转换状态'] = '转换失败'
        else:
            type_log[col]['转换状态'] = '已是数值型'
            type_log[col]['转换后类型'] = str(original_type)
    
    # 保存处理后的数据
    df.to_csv(file)
    type_check_results[code] = type_log

# 展示检查结果
print("\n=== 数据类型检查结果 ===")
for code, results in type_check_results.items():
    print(f"\n{code}:")
    for col, info in results.items():
        print(f"  {col}: {info}")

# 汇总转换情况
conversion_summary = []
for code, results in type_check_results.items():
    for col, info in results.items():
        if info['转换状态'] == '已转换':
            conversion_summary.append({
                '股票代码': code,
                '列名': col,
                '转换情况': f"从{info['原始类型']}转换为{info['转换后类型']}",
                '转换失败数量': info.get('转换失败数量', 0)
            })

if conversion_summary:
    print("\n=== 数据类型转换记录 ===")
    print(pd.DataFrame(conversion_summary))

# %% [markdown]
# **清洗说明 - 数据类型检查前后变化：**
# 
# 清洗前：价格和成交量列可能存在字符串类型（如带逗号的数字、特殊字符等）
# 
# 清洗后：
# - 所有价格、成交量列已统一为数值类型（float64或int64）
# - 无法转换的值被设为NaN，后续需要处理

# %% [markdown]
# ## 5. 重复值处理

# %%
duplicate_stats = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file, index_col='date', parse_dates=True)
    
    # 检测重复行
    duplicate_before = df.duplicated().sum()
    
    if duplicate_before > 0:
        # 删除重复行
        df = df.drop_duplicates()
        deleted_duplicates = duplicate_before
    else:
        deleted_duplicates = 0
    
    duplicate_stats[code] = {
        '重复行数': duplicate_before,
        '删除重复数': deleted_duplicates,
        '剩余行数': len(df)
    }
    
    # 保存处理后的数据
    df.to_csv(file)
    print(f"{code}: 检测到 {duplicate_before} 个重复行，已删除 {deleted_duplicates} 行")

duplicate_df = pd.DataFrame(duplicate_stats).T
print("\n=== 重复值处理统计 ===")
print(duplicate_df)
print(f"\n总计删除重复行数: {duplicate_df['删除重复数'].sum()}")

# %% [markdown]
# **清洗说明 - 重复值处理前后变化：**
# 
# 清洗前：由于数据源问题，可能存在完全相同的重复记录行
# 
# 清洗后：
# - 所有重复行已被删除
# - 保留了第一次出现的记录

# %% [markdown]
# ## 6. 离群值标注

# %%
extreme_stats = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file, index_col='date', parse_dates=True)
    
    # 计算日收益率 (收盘价)
    df['return'] = df['close'].pct_change()
    
    # 标注极端值（涨跌幅超过 ±20%）
    df['is_extreme'] = np.abs(df['return']) > 0.20
    
    # 统计极端值数量
    extreme_count = df['is_extreme'].sum()
    extreme_stats[code] = {
        '极端值数量': extreme_count,
        '极端值比例(%)': (extreme_count / len(df)) * 100
    }
    
    # 删除临时收益率列（按要求不保留）
    df = df.drop('return', axis=1)
    
    # 保存处理后的数据
    df.to_csv(file)
    print(f"{code}: 标注了 {extreme_count} 个极端值记录")

print("\n=== 离群值标注统计 ===")
extreme_df = pd.DataFrame(extreme_stats).T
print(extreme_df)

print("\n=== 离群值可能成因说明 ===")
print("""
1. 重大利空/利好事件：公司突发重大负面新闻或超预期业绩
2. 复牌补涨/补跌：长期停牌后复牌，累积涨跌幅一次性释放
3. 除权除息：股票分红、送股等导致的价格调整（如未复权）
4. 数据错误：原始数据录入或传输错误
5. 极端行情：市场整体剧烈波动（如股灾、熔断）
6. 流动性问题：小盘股因流动性不足导致的异常波动
""")

# %% [markdown]
# **清洗说明 - 离群值标注前后变化：**
# 
# 清洗前：数据中存在极端涨跌幅，可能影响统计分析和模型训练
# 
# 清洗后：
# - 添加了 is_extreme 列，标注了所有单日涨跌幅超过 ±20% 的记录
# - 未删除这些记录，便于后续灵活处理

# %% [markdown]
# ## 7. 合并收盘价为宽表并转换回长表

# %%
# 读取所有股票的收盘价
close_data = {}

for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file, index_col='date', parse_dates=True)
    close_data[code] = df['close']

# 合并为宽表（日期为索引，每列一只股票）
wide_df = pd.DataFrame(close_data)
print(f"\n宽表形状: {wide_df.shape}")
print(f"日期范围: {wide_df.index.min()} 至 {wide_df.index.max()}")
print("\n宽表前5行:")
print(wide_df.head())

# 转换为长表
long_df = wide_df.reset_index().melt(
    id_vars=['date'], 
    var_name='code', 
    value_name='close'
)
# 按日期和代码排序
long_df = long_df.sort_values(['date', 'code']).reset_index(drop=True)

print(f"\n长表形状: {long_df.shape}")
print("\n长表前10行:")
print(long_df.head(10))

# %% [markdown]
# **清洗说明 - 宽表与长表转换前后变化：**
# 
# 清洗前：每只股票的收盘价存储在独立的文件中，不便进行跨股票分析
# 
# 清洗后：
# - 宽表：便于观察股票间的相关性、进行矩阵运算
# - 长表：便于与指数、宏观数据进行合并

# %% [markdown]
# ## 8. 与指数数据合并

# %%
# 读取指数数据
index_files = glob.glob("data/index/index_*.csv")
print(f"找到 {len(index_files)} 个指数文件")

if index_files:
    index_data_list = []
    for file in index_files:
        index_code = file.split("index_")[-1].replace(".csv", "")
        df = pd.read_csv(file)
        
        # 转换日期格式
        df['date'] = pd.to_datetime(df['date'])
        
        # 重命名收盘价列
        df = df[['date', 'close']].rename(columns={'close': f'{index_code}_close'})
        
        index_data_list.append(df)
    
    # 合并所有指数数据
    index_df = index_data_list[0]
    for df in index_data_list[1:]:
        index_df = index_df.merge(df, on='date', how='outer')
    
    print(f"\n指数数据形状: {index_df.shape}")
    print("指数数据前5行:")
    print(index_df.head())
    
    # 与股票长表合并
    print(f"\n合并前股票数据行数: {len(long_df)}")
    print(f"合并前指数数据行数: {len(index_df)}")
    
    # Left join: 保留所有股票数据
    merged_df = long_df.merge(index_df, on='date', how='left')
    
    print(f"\n合并后数据行数: {len(merged_df)}")
    print(f"行数变化: {len(merged_df) - len(long_df)} (增加了指数列，行数不变)")
    
    # 检查缺失的指数数据
    index_cols = [col for col in merged_df.columns if 'close' in col and col != 'close']
    if index_cols:
        missing_index = merged_df[index_cols].isnull().sum()
        if missing_index.sum() > 0:
            print("\n指数数据缺失情况:")
            print(missing_index)
else:
    print("未找到指数数据文件，将跳过指数合并")
    merged_df = long_df.copy()

print("\n合并后数据前10行:")
print(merged_df.head(10))

# %% [markdown]
# **清洗说明 - 与指数数据合并前后变化：**
# 
# 清洗前：个股数据与指数数据分离，无法进行相对收益分析
# 
# 清洗后：
# - 使用 left join，保留了所有个股记录
# - 为每个交易日添加了对应的指数收盘价
# - 合并后行数不变

# %% [markdown]
# ## 9. 与宏观数据合并

# %%
# 读取宏观数据
macro_files = glob.glob("data/macro/macro_*.csv")
print(f"找到 {len(macro_files)} 个宏观数据文件")

if macro_files:
    macro_data = {}
    for file in macro_files:
        macro_name = file.split("macro_")[-1].replace(".csv", "")
        df = pd.read_csv(file)
        
        # 处理日期格式：类似"2026年02月份"的格式
        def parse_macro_date(date_str):
            # 移除"月份"字样，将"年"替换为"-"
            date_str = str(date_str).replace("年", "-").replace("月份", "")
            return pd.to_datetime(date_str, format='%Y-%m')
        
        df['date'] = df['date'].apply(parse_macro_date)
        
        # 获取同比增速列名（除去date列）
        value_col = [col for col in df.columns if col != 'date'][0]
        macro_data[macro_name] = df[['date', value_col]].rename(
            columns={value_col: macro_name}
        )
    
    # 合并宏观数据
    macro_df = None
    for macro_name, df in macro_data.items():
        if macro_df is None:
            macro_df = df
        else:
            macro_df = macro_df.merge(df, on='date', how='outer')
    
    print(f"\n宏观数据形状: {macro_df.shape}")
    print("宏观数据:")
    print(macro_df)
    
    # 将月度宏观数据映射到日度数据
    merged_df['year_month'] = merged_df['date'].dt.to_period('M')
    macro_df['year_month'] = macro_df['date'].dt.to_period('M')
    
    # 执行合并 - 避免列名重复
    macro_for_merge = macro_df.drop('date', axis=1).copy()
    macro_for_merge = macro_for_merge.rename(columns={
        col: f'macro_{col}' for col in macro_for_merge.columns if col != 'year_month'
    })
    
    # 执行合并
    final_df = merged_df.merge(macro_for_merge, on='year_month', how='left')
    
    # 删除辅助列
    final_df = final_df.drop('year_month', axis=1)
    
    print(f"\n合并宏观数据前行数: {len(merged_df)}")
    print(f"合并宏观数据后行数: {len(final_df)}")
    print(f"行数变化: {len(final_df) - len(merged_df)} (行数不变，增加了宏观列)")
    
    # 检查宏观数据缺失
    macro_cols = [col for col in final_df.columns if 'macro_' in col]
    if macro_cols:
        missing_macro = final_df[macro_cols].isnull().sum()
        print("\n宏观数据缺失情况:")
        print(missing_macro)
else:
    print("未找到宏观数据文件，将跳过宏观合并")
    final_df = merged_df.copy()

# 添加股票原有的is_extreme列
print("\n正在添加is_extreme列...")
extreme_data = {}
for file in stock_files:
    code = file.split("stock_")[-1].replace(".csv", "")
    df = pd.read_csv(file, index_col='date', parse_dates=True)
    if 'is_extreme' in df.columns:
        extreme_data[code] = df['is_extreme']

if extreme_data:
    extreme_wide = pd.DataFrame(extreme_data)
    extreme_long = extreme_wide.reset_index().melt(
        id_vars=['date'], 
        var_name='code', 
        value_name='is_extreme'
    )
    # 合并到最终数据
    final_df = final_df.merge(extreme_long, on=['date', 'code'], how='left')
    print(f"已添加is_extreme列，极端值数量: {final_df['is_extreme'].sum()}")

print("\n最终数据前10行:")
print(final_df.head(10))
print(f"\n最终数据形状: {final_df.shape}")
print(f"最终数据列名: {final_df.columns.tolist()}")

# %% [markdown]
# **清洗说明 - 与宏观数据合并前后变化：**
# 
# 清洗前：宏观数据为月度频率，与日度股票数据频率不匹配
# 
# 清洗后：
# - 通过创建年月辅助列，将月度宏观数据映射到对应月份的每个交易日
# - 使用 left join 保留了所有日度数据
# - 合并后行数不变，列数增加

# %% [markdown]
# ## 10. 保存最终结果

# %%
# 保存为CSV格式
final_df.to_csv("data/clean/stock_clean.csv", index=False)
print(f"CSV文件已保存: data/clean/stock_clean.csv")
print(f"文件大小: {os.path.getsize('data/clean/stock_clean.csv') / 1024:.2f} KB")

# 保存为Parquet格式
final_df.to_parquet("data/clean/stock_clean.parquet", index=False)
print(f"Parquet文件已保存: data/clean/stock_clean.parquet")
print(f"文件大小: {os.path.getsize('data/clean/stock_clean.parquet') / 1024:.2f} KB")

# 记录合并过程的行数变化（直接显示，不保存）
print("\n=== 合并过程行数变化记录 ===")
merge_log = pd.DataFrame({
    '步骤': ['股票数据(长表)', '指数合并后', '宏观合并后'],
    '行数': [len(long_df), len(merged_df), len(final_df)],
    '列数': [len(long_df.columns), len(merged_df.columns), len(final_df.columns)],
    '变化说明': [
        '原始股票收盘价长表',
        f'left join指数数据，行数不变，列数增加{len(merged_df.columns) - len(long_df.columns)}' if index_files else '未合并指数数据',
        f'left join月度宏观数据，行数不变，列数增加{len(final_df.columns) - len(merged_df.columns)}' if macro_files else '未合并宏观数据'
    ]
})
print(merge_log)

# %% [markdown]
# ## 11. Parquet格式特性验证

# %%
# 验证Parquet格式的优势
print("\n=== Parquet格式性能对比 ===")

# 1. 列式读取测试（只加载需要的列）
t0 = time.time()
df_subset = pd.read_parquet("data/clean/stock_clean.parquet",
                            columns=["date", "code", "close"])
print(f"列式读取耗时: {time.time()-t0:.3f}s")
print(f"只加载了 {df_subset.shape[1]} 列（原文件共 {final_df.shape[1]} 列）")
print(df_subset.head())

# 2. 查看Schema（类型契约）
import pyarrow.parquet as pq
schema = pq.read_schema("data/clean/stock_clean.parquet")
print(f"\nParquet Schema（类型契约）:")
for name in schema.names:
    field = schema.field(name)
    print(f"  {name}: {field.type}")

# 3. 性能对比：CSV vs Parquet
# CSV读取
t0 = time.time()
df_csv = pd.read_csv("data/clean/stock_clean.csv")
csv_time = time.time() - t0
csv_size = os.path.getsize('data/clean/stock_clean.csv') / 1024

# Parquet读取
t0 = time.time()
df_parquet = pd.read_parquet("data/clean/stock_clean.parquet")
parquet_time = time.time() - t0
parquet_size = os.path.getsize('data/clean/stock_clean.parquet') / 1024

print(f"\n=== 性能对比结果 ===")
print(f"CSV    读取耗时: {csv_time:.3f}s  文件大小: {csv_size:.1f} KB")
print(f"Parquet读取耗时: {parquet_time:.3f}s  文件大小: {parquet_size:.1f} KB")
print(f"速度提升: {(csv_time/parquet_time):.2f}x")
print(f"空间节省: {(1 - parquet_size/csv_size)*100:.1f}%")

# 验证数据一致性
print(f"\n数据一致性验证:")
print(f"CSV形状: {df_csv.shape}")
print(f"Parquet形状: {df_parquet.shape}")
print(f"内容相同: {df_csv.equals(df_parquet)}")

# %% [markdown]
# ## 清洗流程总结

# %%
print("\n=== 数据清洗流程总结 ===")
print("""
1. 缺失值检测：已完成，统计了所有缺失值
2. 缺失值处理：使用前向填充，删除了严重缺失的记录
3. 日期格式统一：所有日期已转为datetime64并设为索引
4. 数据类型检查：价格和成交量列已统一为数值类型
5. 重复值处理：已删除所有重复行
6. 离群值标注：添加is_extreme列，标注了涨跌幅超过±20%的记录
7. 收盘价合并：合并为宽表后又转换为长表
8. 指数数据合并：left join添加了指数数据
9. 宏观数据合并：将月度数据映射到日度数据
10. 结果保存：同时保存为CSV和Parquet格式
""")

print(f"\n最终数据统计:")
print(f"  总记录数: {len(final_df):,}")
print(f"  股票数量: {final_df['code'].nunique()}")
print(f"  日期范围: {final_df['date'].min()} 至 {final_df['date'].max()}")
if 'is_extreme' in final_df.columns:
    print(f"  极端值数量: {final_df['is_extreme'].sum():,}")

print("\n✅ 所有清洗步骤已完成！")

找到 10 个股票文件

=== 缺失值检测结果 ===
Empty DataFrame
Columns: [列名, 缺失数量, 缺失比例(%)]
Index: []

清洗前总缺失值数量: 0
000002: 删除了 0 行，填充了 0 个缺失值
000858: 删除了 0 行，填充了 0 个缺失值
600036: 删除了 0 行，填充了 0 个缺失值
600048: 删除了 0 行，填充了 0 个缺失值
600050: 删除了 0 行，填充了 0 个缺失值
600104: 删除了 0 行，填充了 0 个缺失值
600233: 删除了 0 行，填充了 0 个缺失值
600519: 删除了 0 行，填充了 0 个缺失值
601857: 删除了 0 行，填充了 0 个缺失值
601898: 删除了 0 行，填充了 0 个缺失值

=== 缺失值处理总结 ===
        原始行数  清洗后行数  删除行数  填充缺失值数量
000002  1512   1512     0        0
000858  1512   1512     0        0
600036  1512   1512     0        0
600048  1512   1512     0        0
600050  1512   1512     0        0
600104  1512   1512     0        0
600233  1512   1512     0        0
600519  1512   1512     0        0
601857  1512   1512     0        0
601898  1512   1512     0        0
000002: 日期格式已转换为 datetime64，并设为索引
000858: 日期格式已转换为 datetime64，并设为索引
600036: 日期格式已转换为 datetime64，并设为索引
600048: 日期格式已转换为 datetime64，并设为索引
600050: 日期格式已转换为 datetime64，并设为索引
600104: 日期格式已转换为 datetime64，并设为索引
600233: 日期格式已转换为 datetime6

### 3.1 单表清洗   
- 未有缺失值和离群值   
### 3.2 宽表与长表转换   
- 宽表（每行日期、每列股票）适合横截面分析、相关性矩阵计算、投资组合优化、滚动窗口运算和一次性可视化多只股票对比；
- 长表（每行是日期-股票组合）适合按股票分组操作（如计算单只股票的累计收益率）、与指数/宏观数据合并、数据库存储、以及基于股票代码的筛选和聚合（如计算每只股票的平均成交量）。
### 3.3 多表合并   
- 行数变化见结果输出   
- 两种格式对比（详细差别见输出结果）：  
   - 在本次数据规模（约1MB级别）下，Parquet相比CSV速度提升近5倍、体积减少82.5%，差异已经非常显著；而在更大数据量（如千万行以上）、只读取部分列、网络传输或频繁读写等场景下，Parquet的列式存储、高效压缩和类型保留特性会使其优势进一步扩大到10-50倍，差异将更为悬殊。   
### 提示词：   
- 我在当前目录下的data/stock文件夹中有十个csv，类似stock_000002这样后面是股票代码的命名方式，其中储存了对应股票日度的各种数据，每列标题分别是date，open，high，low，close，volume，	amount（含义为日期、开盘价、收盘价、最高价、最低价、成交量、成交额），我在使用VScode，我想在 02_clean.ipynb 中，完成以下清洗步骤，每步须有文字说明清洗前后的变化（不能只有代码）：
缺失值检测，统计每列缺失值的数量和比例，制成表格；
缺失值处理，向前填充（ffill）或删除，须说明选择依据。
日期格式统一，确保所有表的日期列统一为 datetime64 格式，并设为索引。
数据类型检查，确认价格、成交量列为数值型；若存在字符型需转换并记录。
重复值处理，检测并删除重复行，记录删除数量。
离群值标注，计算日收益率，对单日涨跌幅超过 ±20% 的记录，在新列 is_extreme 中标注为 True（不删除，但须说明可能成因），此步只标出离群值，不需添加收益率新列。
上述过程直接对原csv进行操作。
将 10 只股票的收盘价合并为宽表（日期为索引，每列一只股票），再用 pd.melt 转换回长表，字段为 date, code, close。
我在data/index文件夹中有两个csv，类似index_000300这样后面是指数代码的命名方式，其中储存了对应指数日度的各种数据，每列标题分别是date，open，high，low，close，volume，amount（含义为日期、开盘价、收盘价、最高价、最低价、成交量、成交额）。
将上文长表个股日度数据与指数日度数据按日期做 left join。
我在data/macro文件夹中有两个csv，类似macro_cpi，macro_m2这样后面是宏观指标名称的命名方式，其中储存了对应指标月度的同比增速，每列标题分别是date，cpi_yoy（含义为日期、cpi同比增速），date，m2_yoy（含义为日期、m2同比增速），注意这两个文件中日期的格式类似2026年02月份。
将月度宏观数据与上文合并日度数据再次合并（需处理频率不一致问题：将宏观数据映射到对应月份的每个交易日，或合并至月度收益率数据中）。
将最终结果保存在data/clean文件夹的stock_clean.csv文件中，记录每次合并前后的行数，说明行数变化的原因。
将清洗后的数据额外保存一份 Parquet 格式，存于 data/clean的stock_clean.parquet文件中，并体现以下特性
import pandas as pd, pyarrow.parquet as pq, os, time

列式读取（只加载需要的列）
df = pd.read_parquet("data/clean/stock_clean.parquet",
                     columns=["date", "code", "close"])

查看 Schema（类型契约）
schema = pq.read_schema("data/clean/stock_clean.parquet")
print(schema)

与 CSV 对比：读取速度与文件体积
t0 = time.time()
pd.read_csv("data/clean/stock_clean.csv")
print(f"CSV  读取耗时: {time.time()-t0:.3f}s  "
      f"文件大小: {os.path.getsize('data/clean/stock_clean.csv')/1024:.1f} KB")

t0 = time.time()
pd.read_parquet("data/clean/stock_clean.parquet")
print(f"Parquet 读取耗时: {time.time()-t0:.3f}s  "
      f"文件大小: {os.path.getsize('data/clean/stock_clean.parquet')/1024:.1f} KB")
给我完整的代码，不要分开给

